# Knapsack problem
This is a very common combinatorial optimisation problem where you are given a knapsack of a given capacity $C$, a bunch of items with values and size. The goal is to fill the knapsack with the best aggregated value, respecting the size constraint.

In [ ]:
# patching asyncio so that applications using async functions can run in jupyter
import nest_asyncio
nest_asyncio.apply()
import logging
logging.basicConfig(level=logging.INFO)
import time
from pprint import pprint

In [ ]:
import sys, os
print(os.getcwd())
sys.path.append('../')
sys.path.append('../discrete_optimization')
from discrete_optimization.knapsack.knapsack_model import KnapsackModel, KnapsackSolution
from discrete_optimization.knapsack.solvers import cp_solvers, lp_solvers, greedy_solvers

In [ ]:
from discrete_optimization.knapsack.knapsack_parser import get_data_available, parse_file

## Parse one instance of knapsack


In [ ]:
files_available = get_data_available()
model_file = [f for f in files_available if "ks_60_0" in f][0]
model: KnapsackModel = parse_file(model_file, force_recompute_values=True)
solution = model.get_dummy_solution()

In [ ]:
print(model)

In [ ]:
print(solution)

## Greedy solver

In [ ]:
greedy_solver = greedy_solvers.GreedyBest(knapsack_model=model)

In [ ]:
results = greedy_solver.solve()

In [ ]:
best_result = results.get_best_solution()
print(best_result)

# Exact solver

In [ ]:
from discrete_optimization.knapsack.solvers.lp_solvers import LPKnapsack, LPKnapsackCBC, MilpSolverName, \
ParametersMilp
lp_solver = LPKnapsack(knapsack_model=model, milp_solver_name=MilpSolverName.CBC)

In [ ]:
params_milp = ParametersMilp(time_limit=100, 
                             pool_solutions=10000,
                             mip_gap_abs=0.0001, 
                             mip_gap=0.001,
                             retrieve_all_solution=False, 
                             n_solutions_max=10000)
results_cbc = lp_solver.solve(parameters_milp=params_milp)

In [ ]:
lp_solver_gurobi = LPKnapsack(knapsack_model=model, milp_solver_name=MilpSolverName.GRB)
results_gurobi = lp_solver_gurobi.solve(parameters_milp=params_milp)

In [ ]:
print(results_cbc.get_best_solution(),'\n', results_gurobi.get_best_solution())

## Big problem 

In [ ]:
model_file = [f for f in files_available if "ks_500_0" in f][0]  # optim result "54939"
big_model: KnapsackModel = parse_file(model_file)

In [ ]:
from discrete_optimization.knapsack.solvers.cp_solvers import CPKnapsackMZN2

In [ ]:
lp_solver_gurobi = LPKnapsack(knapsack_model=big_model, milp_solver_name=MilpSolverName.GRB)
results_gurobi = lp_solver_gurobi.solve(parameters_milp=params_milp)

## LNS

In [ ]:
from discrete_optimization.generic_tools.cp_tools import ParametersCP, CPSolverName
from discrete_optimization.knapsack.solvers.cp_solvers import CPKnapsackMZN2, KnapsackModel
from discrete_optimization.generic_tools.do_problem import get_default_objective_setup
from discrete_optimization.knapsack.knapsack_parser import files_available, parse_file
from discrete_optimization.knapsack.solvers.knapsack_lns_cp_solver import ConstraintHandlerKnapsack, \
    InitialKnapsackMethod, InitialKnapsackSolution
from discrete_optimization.generic_tools.lns_cp import LNS_CP

In [ ]:
params_objective_function = get_default_objective_setup(problem=model)
params_cp = ParametersCP.default()
params_cp.TimeLimit = 10
params_cp.TimeLimit_iter0 = 1
solver = CPKnapsackMZN2(big_model,
                        cp_solver_name=CPSolverName.CPOPT,
                        params_objective_function=params_objective_function)
solver.init_model()
initial_solution_provider = InitialKnapsackSolution(problem=big_model,
                                                    initial_method=InitialKnapsackMethod.DUMMY,
                                                    params_objective_function=params_objective_function)
constraint_handler = ConstraintHandlerKnapsack(problem=big_model,
                                               fraction_to_fix=0.8)
lns_solver = LNS_CP(problem=big_model,
                    cp_solver=solver,
                    initial_solution_provider=initial_solution_provider,
                    constraint_handler=constraint_handler,
                    params_objective_function=params_objective_function)
result_store = lns_solver.solve_lns(parameters_cp=params_cp,
                                    nb_iteration_lns=200, nb_iteration_ln)